In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras.datasets import mnist

(X_train, _), (X_test, _) = mnist.load_data()
X_train_np = X_train.reshape(X_train.shape[0], -1).astype(np.float32) / 255.0
X_test_np = X_test.reshape(X_test.shape[0], -1).astype(np.float32) / 255.0
X_train_np.shape

In [ ]:
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error

pca = PCA(n_components=32, random_state=42)
X_train_pca = pca.fit_transform(X_train_np)
X_test_pca = pca.transform(X_test_np)
X_test_pca_recon = pca.inverse_transform(X_test_pca)
pca_recon_mse = mean_squared_error(X_test_np, X_test_pca_recon)

{
    "pca_components": int(pca.n_components_),
    "pca_explained_variance_ratio_sum": float(np.sum(pca.explained_variance_ratio_)),
    "pca_reconstruction_mse": float(pca_recon_mse)
}

In [ ]:
import tensorflow as tf
from tensorflow.keras import Model, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense

tf.keras.utils.set_random_seed(42)
input_dim = X_train_np.shape[1]
latent_dim = 32

inputs = Input(shape=(input_dim,))
x = Dense(256, activation="relu")(inputs)
x = Dense(128, activation="relu")(x)
latent = Dense(latent_dim, activation="linear", name="latent")(x)
x = Dense(128, activation="relu")(latent)
x = Dense(256, activation="relu")(x)
outputs = Dense(input_dim, activation="linear")(x)

autoencoder = Model(inputs, outputs)
autoencoder.compile(optimizer="adam", loss="mse")

early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

history = autoencoder.fit(
    X_train_np,
    X_train_np,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

X_test_ae_recon = autoencoder.predict(X_test_np, verbose=1)
ae_recon_mse = mean_squared_error(X_test_np, X_test_ae_recon)

{
    "autoencoder_latent_dim": latent_dim,
    "autoencoder_best_val_loss": float(np.min(history.history["val_loss"])),
    "autoencoder_reconstruction_mse": float(ae_recon_mse)
}

In [ ]:
comparison = pd.DataFrame([
    {
        "method": "PCA",
        "latent_dim": int(pca.n_components_),
        "reconstruction_mse": float(pca_recon_mse)
    },
    {
        "method": "Autoencoder",
        "latent_dim": latent_dim,
        "reconstruction_mse": float(ae_recon_mse)
    }
])
comparison.sort_values(by=["reconstruction_mse"]).reset_index(drop=True)

In [ ]:
import matplotlib.pyplot as plt

n = 8
plt.figure(figsize=(16, 6))
for i in range(n):
    ax = plt.subplot(3, n, i + 1)
    plt.imshow(X_test_np[i].reshape(28, 28), cmap="gray")
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    if i == 0:
        ax.set_title("Original")
        
    ax = plt.subplot(3, n, i + 1 + n)
    plt.imshow(X_test_ae_recon[i].reshape(28, 28), cmap="gray")
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    if i == 0:
        ax.set_title("Autoencoder")
        
    ax = plt.subplot(3, n, i + 1 + 2 * n)
    plt.imshow(X_test_pca_recon[i].reshape(28, 28), cmap="gray")
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    if i == 0:
        ax.set_title("PCA")

plt.show()